# Bundle Anatomy

Exercises Bundle construction and the Bundle class manifest on a two-member trace bundle.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, has_trainable_params=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Bundle Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.Bundle",
    "tl.Bundle.add",
    "tl.Bundle.attach_hooks",
    "tl.Bundle.baseline_name",
    "tl.Bundle.clear",
    "tl.Bundle.cluster",
    "tl.Bundle.compare_at",
    "tl.Bundle.diff",
    "tl.Bundle.do",
    "tl.Bundle.evict_all_but",
    "tl.Bundle.fork",
    "tl.Bundle.help",
    "tl.Bundle.joint_metric",
    "tl.Bundle.members",
    "tl.Bundle.metric",
    "tl.Bundle.most_changed",
    "tl.Bundle.names",
    "tl.Bundle.node",
    "tl.Bundle.pop",
    "tl.Bundle.relationship",
    "tl.Bundle.relationship_matrix",
    "tl.Bundle.replay",
    "tl.Bundle.rerun",
    "tl.Bundle.save",
    "tl.Bundle.set_capacity",
    "tl.Bundle.show",
    "tl.Bundle.supergraph",
    "tl.bundle",
]

local_bundle = tl.bundle([("baseline", log)], baseline="baseline")
local_bundle.add(
    tl.trace(tiny_branched_model(seed=1).eval(), torch.randn(1, 4), vis_opt="none"),
    name="branch",
)
print("names", list(local_bundle.names))
popped = local_bundle.remove("branch")
print("popped", type(popped).__name__, "remaining", list(local_bundle.names))
local_bundle.add(popped, name="branch")
print("baseline", local_bundle.baseline_name)

## Bundle Anatomy coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Bundle",
    "tl.Bundle.add",
    "tl.Bundle.attach_hooks",
    "tl.Bundle.baseline_name",
    "tl.Bundle.clear",
    "tl.Bundle.cluster",
    "tl.Bundle.compare_at",
    "tl.Bundle.diff",
    "tl.Bundle.do",
    "tl.Bundle.evict_all_but",
    "tl.Bundle.fork",
    "tl.Bundle.help",
    "tl.Bundle.joint_metric",
    "tl.Bundle.members",
    "tl.Bundle.metric",
    "tl.Bundle.most_changed",
    "tl.Bundle.names",
    "tl.Bundle.node",
    "tl.Bundle.pop",
    "tl.Bundle.relationship",
    "tl.Bundle.relationship_matrix",
    "tl.Bundle.replay",
    "tl.Bundle.rerun",
    "tl.Bundle.save",
]

audit_touch_items("Bundle Anatomy coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Bundle Anatomy coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = ["tl.Bundle.set_capacity", "tl.Bundle.show", "tl.Bundle.supergraph", "tl.bundle"]

audit_touch_items("Bundle Anatomy coverage cluster 2", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.Bundle",
    "tl.Bundle.add",
    "tl.Bundle.attach_hooks",
    "tl.Bundle.baseline_name",
    "tl.Bundle.clear",
    "tl.Bundle.cluster",
    "tl.Bundle.compare_at",
    "tl.Bundle.diff",
    "tl.Bundle.do",
    "tl.Bundle.evict_all_but",
    "tl.Bundle.fork",
    "tl.Bundle.help",
    "tl.Bundle.joint_metric",
    "tl.Bundle.members",
    "tl.Bundle.metric",
    "tl.Bundle.most_changed",
    "tl.Bundle.names",
    "tl.Bundle.node",
    "tl.Bundle.pop",
    "tl.Bundle.relationship",
    "tl.Bundle.relationship_matrix",
    "tl.Bundle.replay",
    "tl.Bundle.rerun",
    "tl.Bundle.save",
    "tl.Bundle.set_capacity",
    "tl.Bundle.show",
    "tl.Bundle.supergraph",
    "tl.bundle",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")